# Paper Trading Loop

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ivikasavnish/algo-trading-notebooks/blob/main/notebooks/17_paper_trading_loop.ipynb)

An SMA-crossover signal tracked as simulated paper trades.

Part 17 of 20 in the [ServLoci algo/options trading notebook series](https://comm.servloci.in/docs) — full index in `notebooks/README.md`.

## Setup

In [ ]:
# Get your dedicated static IPv6 + SOCKS5 credentials free:
#   https://comm.servloci.in/register        (or /auth/google?free=1 for an instant trial)
# Your api_key / api_secret pair shows up in the portal after signup:
#   https://comm.servloci.in/user
!pip install -q "requests[socks]"
!curl -sL https://comm.servloci.in/sdk/servloci.py -o servloci.py

import os
from servloci import ServLoci

SERVLOCI_API_KEY = os.environ.get("SERVLOCI_API_KEY", "dhan:1000000001")   # broker:client_id
SERVLOCI_API_SECRET = os.environ.get("SERVLOCI_API_SECRET", "")            # from the portal — leave blank to run this notebook in demo mode

sl = None
if SERVLOCI_API_SECRET:
    sl = ServLoci(api_key=SERVLOCI_API_KEY, api_secret=SERVLOCI_API_SECRET)
    print("ServLoci configured:", sl.host, sl.port)
else:
    print("SERVLOCI_API_SECRET not set — running in demo mode (no live proxy calls).")

An SMA-crossover signal over historical data (notebook 13), tracked as paper trades — no order dispatch yet, see notebook 18 for that.

In [ ]:
import pandas as pd

nifty = pd.read_csv("nifty_2y.csv", index_col=0, parse_dates=True)
nifty["sma_fast"] = nifty["Close"].rolling(10).mean()
nifty["sma_slow"] = nifty["Close"].rolling(30).mean()
nifty["signal"] = 0
nifty.loc[nifty["sma_fast"] > nifty["sma_slow"], "signal"] = 1
nifty.loc[nifty["sma_fast"] < nifty["sma_slow"], "signal"] = -1
nifty["position_change"] = nifty["signal"].diff().fillna(0)

trade_log = []
for date, row in nifty.dropna(subset=["sma_slow"]).iterrows():
    if row["position_change"] != 0:
        trade_log.append({"date": date, "signal": int(row["signal"]), "price": row["Close"]})

paper_trades = pd.DataFrame(trade_log)
print(f"{len(paper_trades)} paper trades generated")
paper_trades.tail()

---

« Previous: [Order Management System](16_order_management_system.ipynb)  
Next: [Signal-to-Order Pipeline](18_signal_to_order_pipeline.ipynb) »

Try the concepts above interactively: [Options Strategy Builder](https://comm.servloci.in/tools/strategy-builder) · [Docs](https://comm.servloci.in/docs) · [Get your static IP](https://comm.servloci.in/register)